 # Shifting small molecule to the center of a non orthorhombic box

 ### Using C60 fullerene in an NU-1000 shaped box as an example
 ### Only showing atoms no bonds, for convenience. 
 ### conda install -c conda-forge ase plotly

In [1]:
import numpy as np 


# ### let's use the box from nu1000 as a non orthorhombic example
from ase.io import read 
from ase.geometry import complete_cell 

data = read('./TestCIFs/nu1000_2.cif')
cell = complete_cell(data.get_cell()).T

# ### let's read in a small molecule this could be from a simulation
molecule= read('./C60.xyz')
coords =molecule.get_positions()


c:\Users\Public\anaconda3\envs\jl_env\lib\site-packages\ase\io\cif.py:401: UserWarning: crystal system 'hexagonal' is not interpreted for space group Spacegroup(191, setting=1). This may result in wrong setting!
  warnings.warn(


In [2]:
def plot_cell(ase_atoms, color='purple', name='cell'):
    """
    It takes an ASE atoms object and returns a plotly scatter3d object that can be used to plot the unit
    cell of the atoms object
    
    Args:
      ase_atoms: the atoms object from ASE
      color: the color of the cell lines. Defaults to purple
      name: the name of the cell. Defaults to cell
    
    Returns:
      A plotly graph object
    """
    
    def draw_cell(A):
        import numpy as np
        avec = A[:, 0].T
        bvec = A[:, 1].T
        cvec = A[:, 2].T
        origin = np.array([0, 0, 0])
        cell_lines = []
        cell_lines.append(origin)
        cell_lines.append(avec)
        cell_lines.append([None, None, None])
        cell_lines.append(origin)
        cell_lines.append(bvec)
        cell_lines.append([None, None, None])
        cell_lines.append(origin)
        cell_lines.append(cvec)
        cell_lines.append([None, None, None])
        cell_lines.append(avec)
        cell_lines.append(avec + bvec)
        cell_lines.append([None, None, None])
        cell_lines.append(bvec)
        cell_lines.append(avec + bvec)
        cell_lines.append([None, None, None])
        cell_lines.append(bvec)
        cell_lines.append(cvec + bvec)
        cell_lines.append([None, None, None])
        cell_lines.append(cvec)
        cell_lines.append(cvec + bvec)
        cell_lines.append([None, None, None])
        cell_lines.append(cvec)
        cell_lines.append(cvec + avec)
        cell_lines.append([None, None, None])
        cell_lines.append(avec)
        cell_lines.append(cvec + avec)
        cell_lines.append([None, None, None])
        cell_lines.append(avec + bvec)
        cell_lines.append(cvec + avec + bvec)
        cell_lines.append([None, None, None])
        cell_lines.append(avec + cvec)
        cell_lines.append(cvec + avec + bvec)
        cell_lines.append([None, None, None])
        cell_lines.append(cvec + bvec)
        cell_lines.append(cvec + avec + bvec)
        cell_lines.append([None, None, None])
        return np.array(cell_lines)

    import plotly.graph_objects as go
    from ase.io import read

    A_unit = ase_atoms.get_cell().T
    cell_lines = draw_cell(A_unit)
    data_cell = go.Scatter3d(z=cell_lines[:, 2], y=cell_lines[:, 1], x=cell_lines[:, 0], mode='lines',
                             line=dict(color=color, width=8), name=name)
    return data_cell

* ### Case 1: The coordinates are like this

In [3]:
import plotly.graph_objects as go 
before = go.Scatter3d(x=coords[:,0], y=coords[:,1], z=coords[:,2], marker_size=2, mode='markers', name='before')
go.Figure(data=[before, plot_cell(data)]).update_layout(template='simple_white',scene = dict(aspectmode='data')).show()


 ### get fractional coordinates for benzene in box, multiplying with inverse cell matrix

In [4]:
coords_frac = np.dot(np.linalg.inv(cell), coords.T).T


 ### compute the centroid with  pbc

In [5]:
# periodic_center = circmean(coords_frac, low=0, high=1, axis=0)
center =np.mean(coords_frac, axis=0)
center_you_want= [0.5,0.5,0.5]

 ### now we are ready to shift by center_you_want  - periodci center

In [6]:
shifted_coords = coords_frac + (center_you_want-center)

 ### lets convert everything back to cartesian

In [7]:
coords_cart = np.dot(cell,coords.T).T
shifted_coords_cart = np.dot(cell, shifted_coords.T).T

 ### Plot everything to confirm

In [8]:
import plotly.graph_objects as go 
after = go.Scatter3d(x=shifted_coords_cart[:,0], y=shifted_coords_cart[:,1], z=shifted_coords_cart[:,2], marker_size =2, mode='markers+lines', name='after')
go.Figure(data=[before, after, plot_cell(data)]).update_layout(template='simple_white',scene = dict(aspectmode='data')).show()

 ## Case 2: MOlecule is all broken up like this

In [9]:
temp = coords_frac# + ([0.5,0.5,0.5]-np.mean(coords_b_frac, axis=0))
temp = temp - np.floor(temp)
coords_broken= np.dot(cell,temp.T).T

import plotly.graph_objects as go 
before = go.Scatter3d(x=coords_broken[:,0], y=coords_broken[:,1], z=coords_broken[:,2], marker_size=2, mode='markers', name='before')
go.Figure(data=[before, plot_cell(data)]).update_layout(template='simple_white',scene = dict(aspectmode='data')).show()

 ### we have to rebuild the molecule from its parts !!

  ### first we put together the molecule in fractional coordinates in the top left corner using np.round

In [10]:

coords_broken_frac = np.dot(np.linalg.inv(cell), coords_broken.T).T
coords_broken_frac = coords_broken_frac - np.round(coords_broken_frac)
coords_reconstruct = np.dot(cell, coords_broken_frac.T).T

import plotly.graph_objects as go 
intermediate = go.Scatter3d(x=coords_reconstruct[:,0], y=coords_reconstruct[:,1], z=coords_reconstruct[:,2], mode='markers',marker_size=2,  name='intermediate')
# go.Figure(data=[intermediate, plot_cell(data)]).update_layout(template='simple_white',scene = dict(aspectmode='data')).show()



 ### now repeat the same steps as in Case 1

In [11]:
coords_reconstruct_frac = np.dot(np.linalg.inv(cell),coords_reconstruct.T).T
center=np.mean(coords_reconstruct_frac , axis=0)
center_you_want= [0.5,0.5,0.5]
shifted_coords_reconstruct_frac = coords_reconstruct_frac    + (center_you_want-center)
shifted_coords_reconstruct_cart = np.dot(cell, shifted_coords_reconstruct_frac.T).T

import plotly.graph_objects as go 
after = go.Scatter3d(x=shifted_coords_reconstruct_cart[:,0], y=shifted_coords_reconstruct_cart[:,1], z=shifted_coords_reconstruct_cart[:,2], marker_size =2, mode='markers', name='after')
go.Figure(data=[before, intermediate, after, plot_cell(data)]).update_layout(template='simple_white',scene = dict(aspectmode='data')).show()
#